# Partie 2 — Tester le modèle Fashion-MNIST

Ce notebook recharge le **modèle et les poids entraînés** par `classification.ipynb`, puis classe une image de vêtement. Le tutoriel propose une image externe `image.png` ; si elle n'est pas fournie, on construit une image d'exemple **à partir du CSV de test non utilisé pendant l'entraînement** pour pouvoir exécuter toutes les cellules.

**Avant l'exécution :** exécuter et sauvegarder `classification.ipynb` afin de créer `modele/modele.json` et `modele/modele.weights.h5`.

In [ ]:
from pathlib import Path
import numpy as np
from PIL import Image, ImageFilter
from tensorflow.keras.models import model_from_json

RACINE = Path.cwd()
if not (RACINE / 'modele' / 'modele.json').exists():
    RACINE = RACINE / 'analyse_image'
assert (RACINE / 'modele' / 'modele.json').exists(), 'Exécutez classification.ipynb avant ce notebook.'
assert (RACINE / 'modele' / 'modele.weights.h5').exists(), 'Les poids du modèle sont manquants.'

with open(RACINE / 'modele' / 'modele.json', 'r', encoding='utf-8') as fichier_json:
    modele_json = fichier_json.read()
modele = model_from_json(modele_json)
modele.load_weights(str(RACINE / 'modele' / 'modele.weights.h5'))
print('Modèle chargé depuis :', RACINE / 'modele')

## 1. Catégories de classification

In [ ]:
classes = [
    'Un T-shirt/haut', 'Un pantalon', 'Un pull', 'Une robe',
    'Un manteau', 'Des sandales', 'Une chemise', 'Des baskets',
    'Un sac', 'Des bottes de cheville'
]
print('Nombre de catégories :', len(classes))

## 2. Prétraiter une nouvelle image avec PIL

Reprise des fonctions du tutoriel : niveaux de gris, redimensionnement, centrage et normalisation.

In [ ]:
def resize_with_aspect_ratio(image, target_dimension):
    width, height = image.size
    if width > height:
        new_width = target_dimension
        new_height = max(1, round(target_dimension * height / width))
    else:
        new_height = target_dimension
        new_width = max(1, round(target_dimension * width / height))
    return image.resize((new_width, new_height), Image.Resampling.LANCZOS).filter(ImageFilter.SHARPEN)


def create_centered_image(image, canvas_size):
    canvas = Image.new('L', (canvas_size, canvas_size), 255)
    img_width, img_height = image.size
    x_offset = (canvas_size - img_width) // 2
    y_offset = (canvas_size - img_height) // 2
    canvas.paste(image, (x_offset, y_offset))
    return canvas


def normalize_pixels(image):
    pixels = np.array(image.getdata(), dtype=np.float32)
    return (255 - pixels) / 255.0


def load_and_preprocess_image(image_path, target_size=28, content_size=20):
    image = Image.open(image_path).convert('L')
    resized_img = resize_with_aspect_ratio(image, content_size)
    canvas = create_centered_image(resized_img, target_size)
    normalized_array = normalize_pixels(canvas)
    return normalized_array.reshape(1, target_size, target_size, 1)

print('Fonctions de prétraitement définies.')

## 3. Choisir l’image à tester

- Si `analyse_image/images/image.png` existe, elle sera utilisée (vêtement foncé sur fond clair, comme dans le tutoriel).
- Sinon, une image d’exemple est générée à partir d’une observation du fichier `fashion-mnist_test.csv`. Elle est enregistrée en PNG avec les couleurs inversées pour pouvoir passer par les mêmes fonctions PIL du tutoriel.

In [ ]:
image_path = RACINE / 'images' / 'image.png'
etiquette_attendue = None
if not image_path.exists():
    import pandas as pnd
    donnees_test = pnd.read_csv(RACINE / 'datas' / 'fashion-mnist_test.csv', nrows=1)
    etiquette_attendue = int(donnees_test.iloc[0, 0])
    pixels = donnees_test.iloc[0, 1:].to_numpy(dtype=np.uint8).reshape(28, 28)
    # CSV Fashion-MNIST : vêtement clair sur fond noir ; PIL du tutoriel attend l'inverse
    image_path = RACINE / 'images' / 'exemple_depuis_test.png'
    image_path.parent.mkdir(exist_ok=True)
    Image.fromarray(255 - pixels, mode='L').save(image_path)
    print('Image de test générée :', image_path, '| catégorie attendue :', classes[etiquette_attendue])
else:
    print('Image externe choisie :', image_path)

display(Image.open(image_path))

## 4. Prédire la catégorie de l’image

In [ ]:
image_array = load_and_preprocess_image(image_path)
probabilities = modele.predict(image_array, verbose=0)[0]
predicted_class = int(np.argmax(probabilities))
print(f'Selon moi l’image est : {classes[predicted_class]}\n')
for i, class_name in enumerate(classes):
    print(f'{class_name}: {probabilities[i] * 100:.2f}%')
if etiquette_attendue is not None:
    print('Catégorie réelle dans Fashion-MNIST :', classes[etiquette_attendue])